In [2]:
import pandas as pd

# Load raw data
df = pd.read_csv("../data/accepted_2007_to_2018Q4.csv.gz", low_memory=False)

# Keep only loans with a known outcome
finished = ['Fully Paid', 'Charged Off', 'Default',
            'Does not meet the credit policy. Status:Fully Paid',
            'Does not meet the credit policy. Status:Charged Off']
df = df[df['loan_status'].isin(finished)].copy()

# Create target: 1 = default, 0 = paid
default_statuses = ['Charged Off', 'Default',
                    'Does not meet the credit policy. Status:Charged Off']
df['default_flag'] = df['loan_status'].isin(default_statuses).astype(int)

# Drop leakage columns
leakage_cols = [
    'loan_status',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term',
    'out_prncp', 'out_prncp_inv',
]
leakage_cols += [c for c in df.columns if c.startswith('hardship_')]
df = df.drop(columns=[c for c in leakage_cols if c in df.columns])

# Downsample to ~200k, preserve 80/20 class balance
n_target = 200_000
n_bad = int(n_target * df['default_flag'].mean())   # ~40k defaults
n_good = n_target - n_bad                            # ~160k paid

df_bad = df[df['default_flag'] == 1].sample(n=n_bad, random_state=42)
df_good = df[df['default_flag'] == 0].sample(n=n_good, random_state=42)

df = pd.concat([df_good, df_bad]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final shape: {df.shape}")
print(f"Class balance:\n{df['default_flag'].value_counts(normalize=True).round(3)}")

df.to_csv("../data/loans_clean.csv", index=False)
print("Saved to data/loans_clean.csv")

Final shape: (200000, 117)
Class balance:
default_flag
0    0.8
1    0.2
Name: proportion, dtype: float64
Saved to data/loans_clean.csv
